<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Omit uncertainties for now. Add them later. Focus on getting the Neural net trained first
#1. Optimize Gradient Descent functions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
data_df_2020 = data_df_2020.drop(columns=[0,4]).reset_index(drop=True)
data_df_2020.columns = range(data_df_2020.columns.size)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8
0,1,1,2,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,2,1,3,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,1,2,3,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,0,3,3,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,3,1,4,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [5]:
og_data = data_df_2020.copy()


In [6]:
input = np.array(data_df_2020.iloc[:,[0,1,2,3,7]].values) #Neutrons, Protons, Mass Number, Mass Excess, and Atomic Mass are take as inputs
output = np.array(data_df_2020.iloc[:,[5]].values) #Binding Energy is our output

print(input)
print(output)


[[1.00000000e+00 1.00000000e+00 2.00000000e+00 1.31357229e+04
  1.41017778e+04]
 [2.00000000e+00 1.00000000e+00 3.00000000e+00 1.49498109e+04
  1.60492813e+04]
 [1.00000000e+00 2.00000000e+00 3.00000000e+00 1.49312189e+04
  1.60293220e+04]
 ...
 [1.77000000e+02 1.17000000e+02 2.94000000e+02 1.96397000e+05
  2.10840000e+05]
 [1.76000000e+02 1.18000000e+02 2.94000000e+02 1.99320000e+05
  2.13979000e+05]
 [1.77000000e+02 1.18000000e+02 2.95000000e+02 2.01369000e+05
  2.16178000e+05]]
[[1112.2831 ]
 [2827.2654 ]
 [2572.68044]
 ...
 [7092.     ]
 [7079.     ]
 [7076.     ]]


In [7]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.2)

#Standardize input and output training data for simpler training
scaler_input = StandardScaler()
scaler_output = StandardScaler()
#must have two different scaler functions since input and output have different column dimensions

input_train = scaler_input.fit_transform(input_train)
output_train = scaler_output.fit_transform(output_train)

input_test = scaler_input.transform(input_test)
output_test = scaler_output.transform(output_test)

In [8]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [9]:
training_data = dataset(input_train,output_train)
testing_data = dataset(input_test,output_test)

In [10]:
#Breaking training and testing data into smaller batches
train_dataloader = DataLoader(training_data,batch_size = 8,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 8,shuffle = True)

In [16]:
for i,b in train_dataloader:
  print(i)
  print("===========")
  print(b)
  break

tensor([[ 0.7518,  0.5793,  0.6895, -0.2284,  0.6335],
        [-1.3127, -1.4285, -1.3679, -0.1618,  0.6440],
        [ 0.8665,  0.7585,  0.8304, -0.1107,  0.6520],
        [-0.1428, -0.2812, -0.1982, -1.1089,  0.4956],
        [-0.2575, -0.6397, -0.4096, -0.2402,  0.6317],
        [-0.8539, -0.5680, -0.7478, -0.7734,  0.5482],
        [ 0.5913,  0.8661,  0.7036,  0.0415,  0.6758],
        [-1.1062, -1.4285, -1.2410,  0.3754,  0.7281]], device='cuda:0')
tensor([[ 0.0040],
        [ 0.7401],
        [-0.0608],
        [ 0.6785],
        [ 0.2300],
        [ 0.7046],
        [-0.1515],
        [-0.1710]], device='cuda:0')


In [11]:

class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],64)
    self.layer2 = nn.Linear(64,32)
    self.OutLayer = nn.Linear(32,1) #Output is Binding Energy
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.Relu(self.layer1(X))
    X = self.Relu(self.layer2(X))
    X = self.OutLayer(X)

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [12]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             384
              ReLU-2                   [-1, 64]               0
            Linear-3                   [-1, 32]           2,080
              ReLU-4                   [-1, 32]               0
            Linear-5                    [-1, 1]              33
Total params: 2,497
Trainable params: 2,497
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [13]:
criterion = nn.MSELoss() #Loss function, uses Mean Squared Error
optimizer = Adam(BE_NeuralNet.parameters(), lr = 0.001) #Gradient Descent algorithm (Adam)

In [20]:
for data in train_dataloader:
  inputs, output = data
  print(inputs)
  print(output)
  break

tensor([[-1.5879, -1.4285, -1.5370,  0.5982, -1.7955],
        [-0.7851, -1.1775, -0.9451,  0.2821,  0.7135],
        [ 0.4307,  0.7227,  0.5486, -0.1663,  0.6433],
        [ 1.8529,  1.6548,  1.7887,  2.3891, -1.5150],
        [ 0.1095,  0.1849,  0.1400, -0.9121,  0.5265],
        [-1.1062, -1.0700, -1.1001, -0.7710,  0.5486],
        [ 0.1095,  0.4358,  0.2386, -0.5373,  0.5852],
        [-0.9227, -0.8907, -0.9169, -0.8813,  0.5313]], device='cuda:0')
tensor([[-0.9633],
        [-0.0773],
        [-0.0557],
        [-0.8392],
        [ 0.4000],
        [ 1.1622],
        [ 0.1539],
        [ 1.0331]], device='cuda:0')


In [21]:
total_loss_train_plot = []
total_loss_test_plot = []
total_acc_train_plot = []
total_acc_test_plot = []

epochs = 20
for epoch in range(epochs):
  total_loss_train = 0
  total_acc_test = []
  total_loss_test = 0

  for data in train_dataloader:
    inputs, output = data

    prediction = BE_NeuralNet(inputs) #Prediction by neural network

    batch_loss = criterion(prediction,output) #calculate loss for current batch
    total_loss_train += batch_loss.item()

    batch_loss.backward() #Backpropagation done on each batch of data
    optimizer.step() #Adjust weights of parameters
    optimizer.zero_grad() #Reset gradient for next batch

  avg_loss_train = total_loss_train/len(train_dataloader) #Average training data loss (before backpropagation) over current epoch

  with torch.no_grad(): #Stop tracking gradients of test data (as it is not needed)
    for data in test_dataloader:
      inputs, output = data

      prediction = BE_NeuralNet(inputs)

      batch_loss = criterion(prediction,output)
      total_loss_test += batch_loss.item()

    avg_loss_test = total_loss_test/len(test_dataloader) #Average testing data loss (before backpropagation) over current epoch


  #Takes average accuracy of each epoch and converts to percent
  avg_acc_test = (sum(total_acc_test)/len(total_acc_test))*100


  print(f'''Epoch: {epoch+1} Train_Loss: {avg_loss_train}
        Testing Loss: {avg_loss_test} Testing Accuracy: {avg_acc_test}''')
  print("=="*25)


Epoch: 1 Train_Loss: 0.46721816628097473 Train_Accuracy: 58.60253143310547
        Testing Loss: 0.1958884336852228 Testing Accuracy: 77.9494400024414
Epoch: 2 Train_Loss: 0.3175574233861747 Train_Accuracy: 77.80899047851562
        Testing Loss: 0.14844109231424307 Testing Accuracy: 84.69100952148438
Epoch: 3 Train_Loss: 0.2654487224439783 Train_Accuracy: 82.1278076171875
        Testing Loss: 0.11313887168470184 Testing Accuracy: 87.21910095214844
Epoch: 4 Train_Loss: 0.2332078422365525 Train_Accuracy: 87.42977142333984
        Testing Loss: 0.0789465279765741 Testing Accuracy: 91.2921371459961
Epoch: 5 Train_Loss: 0.20317900530634883 Train_Accuracy: 88.09690856933594
        Testing Loss: 0.07491327307095132 Testing Accuracy: 91.71348571777344
Epoch: 6 Train_Loss: 0.19112356786260196 Train_Accuracy: 88.48314666748047
        Testing Loss: 0.07607412605652365 Testing Accuracy: 92.27527618408203
Epoch: 7 Train_Loss: 0.17948382260653764 Train_Accuracy: 85.70926666259766
        Testing